# 06 — Ekspor hasil untuk web (Orang 2)

Menulis file yang dibaca backend (Kontrak 6) untuk mesin kelompok C:

- `data/results/armada_r{50,100,200}.json` → `GET /api/armada?rasio=…`
- `data/results/mesin/r{50,100,200}/{unit}.json` → `GET /api/mesin/{unit}?rasio=…`

Satu file per rasio, sehingga backend cukup memilih nama file: `f"armada_r{rasio}.json"`. Isi setiap file sama persis dengan bentuk di Kontrak 6. Jalankan setelah notebook 03 dan 04.

In [ ]:
# Colab: jalankan sel ini dulu (repo, branch, dan data mentah).
import os, shutil, sys
from google.colab import drive
drive.mount('/content/drive')
PATH_DRIVE = '/content/drive/MyDrive/Praktikum MLN/Dataset ML'
BRANCH = 'model'
if not os.path.exists('/content/prognosense'):
    !git clone https://github.com/Arsha8Anuga/PrognoSense /content/prognosense
%cd /content/prognosense
!git checkout {BRANCH} 2>/dev/null || git checkout -b {BRANCH}
sys.path.insert(0, '/content/prognosense')
os.makedirs('data/raw', exist_ok=True)
for f in ['train_FD002.txt', 'train_FD004.txt']:
    shutil.copy(f'{PATH_DRIVE}/{f}', f'data/raw/{f}')

## 1. Memuat model dan data kelompok C

Sensor ternormalisasi dan kondisi per siklus dihitung ulang dengan artefak `models/v1/` yang sama dengan server, untuk grafik Detail mesin (`s4`, `s11`, `s15`).

In [ ]:
import json, os
import numpy as np, pandas as pd
from engine.schema import SENSOR_AKTIF
from engine.data import load_and_format
from engine.conditions import assign_kondisi, apply_normalization
from engine.predict import load, state_rl
from engine.features import RUL_CAP
from engine import evaluate as EV

RAW, HASIL, MODEL = 'data/raw', 'data/results', 'models/v1'
bundle = load(MODEL)
assert set(bundle['qtable']) == set(EV.RASIO), 'jalankan 04_qlearning dulu: Q-table belum lengkap'
preds = pd.read_csv(f'{HASIL}/predictions.csv')
C = preds[preds['kelompok'] == 'C'].sort_values(['unit', 'cycle'])

# kondisi dan sensor ternormalisasi per siklus untuk grafik Detail mesin
train = pd.concat([load_and_format(f'{RAW}/train_FD002.txt', 'FD002'),
                   load_and_format(f'{RAW}/train_FD004.txt', 'FD004')], ignore_index=True)
train = train[train['unit'].isin(C['unit'].unique())]
train = apply_normalization(assign_kondisi(train, bundle['kmeans_cond']), bundle['norm_stats'], SENSOR_AKTIF)
SENSOR_TAMPIL = ['s4', 's11', 's15']
C = C.merge(train[['unit', 'cycle', 'kondisi'] + [f'{s}_norm' for s in SENSOR_TAMPIL]], on=['unit', 'cycle'], how='left')
assert C['kondisi'].notna().all()
TAHAP_IDX = {t: i for i, t in enumerate(bundle['rl']['urutan_tahap'])}
print('mesin C:', C['unit'].nunique())

## 2. Menulis armada dan detail mesin

Dasbor menampilkan mesin C "seolah sedang beroperasi": setiap mesin dipotong di siklus acak antara 50% dan 95% umurnya (`seed = 42`).

Satu aturan ditambahkan supaya Dasbor dan Detail mesin tidak saling bertentangan: aksi di Dasbor adalah **rawat** bila agen sudah memilih rawat di siklus itu atau sebelumnya. Kolom `terlambat = true` menandai mesin yang seharusnya sudah dirawat lebih awal. Tanpa aturan ini, Dasbor bisa menampilkan mesin "masih beroperasi" beberapa siklus setelah Detail mesin mengatakan agen sudah merawatnya.

`rul_aktual` di daftar siklus dipotong di 125 supaya sebanding dengan `rul_pred` di grafik; di `keputusan` nilainya tidak dipotong.

In [ ]:
def q_dan_aksi(q_tab, rul, tahap):
    qs = q_tab[state_rl(rul, TAHAP_IDX[tahap], bundle['rl'])]
    return {'lanjut': round(float(qs[0]), 2), 'rawat': round(float(qs[1]), 2)}, ('rawat' if qs[1] > qs[0] else 'lanjut')

rng = np.random.default_rng(42)
potong = {u: int(rng.uniform(0.5, 0.95) * len(g)) for u, g in C.groupby('unit', sort=True)}   # indeks siklus "sekarang"

for r in EV.RASIO:
    q_tab = bundle['qtable'][r]
    data_c = EV.siapkan(C, list(potong), bundle['rl']['batas_bin'])
    os.makedirs(f'{HASIL}/mesin/r{r}', exist_ok=True)
    armada = []
    for u, g in C.groupby('unit', sort=True):
        g = g.reset_index(drop=True)
        i_rawat = EV.idx_agen(data_c[u], q_tab)            # siklus pertama agen memilih rawat (None = tidak pernah)
        siklus = []
        for b in g.itertuples():
            q, aksi = q_dan_aksi(q_tab, b.rul_pred, b.tahap)
            siklus.append({'cycle': int(b.cycle), 'kondisi': int(b.kondisi),
                           'sensor': {s: round(float(getattr(b, f'{s}_norm')), 3) for s in SENSOR_TAMPIL},
                           'tahap': b.tahap, 'rul_pred': round(float(b.rul_pred), 1),
                           'rul_aktual': int(min(b.rul_aktual, RUL_CAP)), 'aksi': aksi, 'q': q})
        mesin = {'unit': u, 'rasio': r, 'umur_aktual': int(g['cycle'].max()),
                 'kondisi_dominan': int(g['kondisi'].mode()[0]),
                 'keputusan': {'siklus_rawat': None if i_rawat is None else int(g.loc[i_rawat, 'cycle']),
                               'rul_aktual_saat_rawat': None if i_rawat is None else int(g.loc[i_rawat, 'rul_aktual']),
                               'gagal': i_rawat is None},
                 'sensor_ditampilkan': SENSOR_TAMPIL, 'siklus': siklus}
        json.dump(mesin, open(f'{HASIL}/mesin/r{r}/{u}.json', 'w'), separators=(',', ':'))

        i = potong[u]
        q, _ = q_dan_aksi(q_tab, g.loc[i, 'rul_pred'], g.loc[i, 'tahap'])
        # "rawat" bila agen sudah memilih rawat di siklus ini atau sebelumnya; konsisten dengan Detail mesin
        aksi = 'rawat' if i_rawat is not None and i_rawat <= i else 'lanjut'
        armada.append({'unit': u, 'siklus': int(g.loc[i, 'cycle']), 'kondisi': int(g.loc[i, 'kondisi']),
                       'tahap': g.loc[i, 'tahap'], 'rul_pred': round(float(g.loc[i, 'rul_pred']), 1),
                       'rul_aktual': int(g.loc[i, 'rul_aktual']), 'aksi': aksi,
                       'terlambat': bool(i_rawat is not None and i_rawat < i), 'q': q})

    armada.sort(key=lambda m: m['rul_pred'])
    df = pd.DataFrame(armada)
    hasil = {'model_version': 'v1', 'rasio': r,
             'ringkasan': {'total': len(df), 'rawat_sekarang': int((df['aksi'] == 'rawat').sum()),
                           'per_tahap': {t: int((df['tahap'] == t).sum()) for t in bundle['rl']['urutan_tahap']},
                           'rata2_rul_pred': round(float(df['rul_pred'].mean()), 1)},
             'mesin': armada}
    json.dump(hasil, open(f'{HASIL}/armada_r{r}.json', 'w'), indent=1)
    print(f"rasio {r}: {hasil['ringkasan']}")

## 3. Pemeriksaan konsistensi

Setiap baris Dasbor dicocokkan dengan `keputusan` di file Detail mesinnya, untuk ketiga rasio.

In [ ]:
for r in EV.RASIO:
    a = json.load(open(f'{HASIL}/armada_r{r}.json'))
    for m in a['mesin']:
        d = json.load(open(f"{HASIL}/mesin/r{r}/{m['unit']}.json"))['keputusan']
        sudah = d['siklus_rawat'] is not None and d['siklus_rawat'] <= m['siklus']
        assert (m['aksi'] == 'rawat') == sudah, f"dasbor dan detail tidak cocok: {m['unit']} rasio {r}"
print('dasbor dan detail mesin konsisten untuk ketiga rasio')